<a href="https://colab.research.google.com/github/F1ameX/2025-ODS-NLP/blob/main/practice_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install and Import Libraries

In [1]:
!pip install --quiet catboost
!pip install --quiet gensim
!pip install --quiet nltk

In [2]:
import os
import re
import pandas as pd
import numpy as np

import nltk
from catboost import Pool, CatBoostClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import WordNetLemmatizer

In [3]:
nltk.download('wordnet')
nltk.download('stopwords')
stop_words = set(nltk.corpus.stopwords.words('russian'))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Read Data

In [4]:
path = '/content/drive/MyDrive/ods-nlp_src/practice_2/'
train_data = pd.read_csv(os.path.join(path, 'train.csv'))
test_data = pd.read_csv(os.path.join(path, 'test.csv'))
print(f'Number of rows and columns in the train data set: {train_data.shape}')
print(f'Number of rows and columns in the test data set: {test_data.shape}')
train_data.head()

Number of rows and columns in the train data set: (48665, 2)
Number of rows and columns in the test data set: (12167, 2)


,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


In [5]:
train_data.groupby('rate').describe()

text                               
      count unique                top freq
rate                                      
1      4138   4130             Грязно    3
2      2410   2407  Отстойный магазин    2
3      6126   6070          Нормально    7
4      9922   9763               Норм   14
5     26069  24804    Хороший магазин  107

## Preparing the data and creating Catboost model

In [6]:
train_data['text'].sample(7)

,text
15346,Отличный магазин
36675,"Выбор не большой,но проходить можно"
33657,Очень вежливый и отзывчивый персонал работает ...
10585,Пятерочка без изысков
13501,Всё по новым стандартам Пятëрочек. И приятный ...
44108,Особого ввбора магазинов продуктоввх в том мик...
5269,"Персонал хамит, захожу только в случае крайне..."


In [7]:
def process_data(df):
    lemmatizer = WordNetLemmatizer()
    df['text'] = df['text'].str.lower()
    df['text'] = df['text'].apply(lambda x: re.sub(r'([,.!?;])', r' \1 ', x))
    df['text'] = df['text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))
    df['text'] = df['text'].apply(lambda x: re.sub(r'[\d]', '', x))
    df['text'] = df['text'].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))

    return df

In [8]:
train_data = process_data(train_data)
test_data = process_data(test_data)

In [9]:
train_data.sample(15)

,rate,text
22341,1,ценники всегда не актуальные если скидка на ка...
46895,5,хороший магазин в шаговой доступности большой ...
35043,5,пользуюсь постоянно выбор всегда хороший распо...
26852,1,не понравилось то что днем ранее пришлось долг...
38356,4,пятерочка отличная но аптеку в ней обходите ст...
46733,5,без очередей
13536,4,магазин не плохой но очереди большие всегда ин...
35443,5,очень вежливый персонал кассы работают всё нра...
18698,4,приличный магазин чистенько есть всё необходим...
42247,5,отличный ассортимент хорошо оборудован зал не ...


In [10]:
X_train = train_data['text']
y_train = train_data['rate']

X_test = test_data['text']


model = CatBoostClassifier(
    iterations = 150,
    depth = 5,
    random_seed = 52,
)

model.fit(
    X_train,
    y_train,
    text_features = [0],
    verbose=True
)

Learning rate set to 0.479263
0:	learn: 1.1045265	total: 1.39s	remaining: 3m 27s
1:	learn: 1.0136408	total: 2.95s	remaining: 3m 38s
2:	learn: 0.9674237	total: 4.76s	remaining: 3m 53s
3:	learn: 0.9394648	total: 6.55s	remaining: 3m 59s
4:	learn: 0.9283808	total: 8.5s	remaining: 4m 6s
5:	learn: 0.9200120	total: 9.54s	remaining: 3m 49s
6:	learn: 0.9157135	total: 10.6s	remaining: 3m 35s
7:	learn: 0.9136866	total: 11.6s	remaining: 3m 25s
8:	learn: 0.9090013	total: 12.7s	remaining: 3m 18s
9:	learn: 0.9070450	total: 13.7s	remaining: 3m 11s
10:	learn: 0.9033949	total: 14.8s	remaining: 3m 6s
11:	learn: 0.8989732	total: 15.9s	remaining: 3m 2s
12:	learn: 0.8977457	total: 16.9s	remaining: 2m 58s
13:	learn: 0.8951320	total: 18.2s	remaining: 2m 56s
14:	learn: 0.8944480	total: 19.8s	remaining: 2m 58s
15:	learn: 0.8932566	total: 21.4s	remaining: 2m 59s
16:	learn: 0.8920883	total: 22.9s	remaining: 2m 58s
17:	learn: 0.8907625	total: 23.9s	remaining: 2m 55s
18:	learn: 0.8893843	total: 24.9s	remaining: 2m 

## Predict

In [11]:
dataset_test = Pool(
    data = X_test,
    text_features = [0]
)

predict_classes = model.predict(dataset_test)
predictions = predict_classes

## Create submission

In [12]:
sample_submission = pd.read_csv(os.path.join(path, 'sample_submission.csv'))
sample_submission['rate'] = predictions
sample_submission.head()

,index,rate
0,0,5
1,1,5
2,2,5
3,3,5
4,4,1


In [13]:
sample_submission.to_csv(os.path.join(path, 'submission.csv'), index=False)